In [ ]:
import json
import pennylane as qp
import pennylane.numpy as np

# The unitary whose spectrum you have to read. You do not need to change this.
def unitary_matrix(phases):
    """Builds a diagonal unitary with the given eigenphases.

    The eigenvalue on computational-basis state |k> is exp(2j * pi * phases[k]),
    so the eigenvectors are the computational-basis states themselves.

    Args:
        phases (list[float]): eigenphases in [0, 1), one per basis state. The length
            is a power of two.

    Returns:
        numpy array: the diagonal unitary matrix.
    """
    return np.diag(np.array([np.exp(2j * np.pi * phase) for phase in phases]))



def read_spectrum(phases, amplitudes, m):
    """Reads every eigenphase present in a superposition, together with its weight,
    and prices the repetitions needed to see the faintest one.

    Args:
        phases (list[float]): the eigenphases of the unitary, passed to unitary_matrix.
        amplitudes (list[float]): a normalised real state on the target wires.
        m (int): the number of estimation wires.

    Returns:
        tuple[list[float], list[float], int]:
            - the eigenphases present, ascending;
            - the weight of each, in the same order;
            - the number of repetitions needed to see the faintest one.
    """


    # Put your code here.
    eigenphases = None
    weights = None
    repetitions = None

    target_wires = int(np.log2(len(phases)))

    dev = qp.device("default.qubit", wires=target_wires+m)

    @qp.qnode(dev)
    def circuit():
        qp.StatePrep(amplitudes, wires=range(target_wires))

        unitary = unitary_matrix(phases)
    
        qp.QuantumPhaseEstimation(unitary, target_wires=range(target_wires), estimation_wires=range(target_wires, target_wires+m))

        return qp.probs(wires=range(target_wires, target_wires+m))

    probs = circuit()

    indices = [i for i in range(len(probs)) if probs[i] >= 1e-9]

    eigenphases = [k/2**m for k in indices]

    weights = [probs[i] for i in indices]

    repetitions = int(np.ceil(np.log(0.01)/np.log(1-np.min(weights))))

    return eigenphases, weights, repetitions